# WSI: сравнение алгоритмов кластеризации

## 1. Подготовка

In [1]:
%pip install -q pandas numpy scikit-learn sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import (
    KMeans,
    DBSCAN,
    AffinityPropagation,
    AgglomerativeClustering,
    SpectralClustering,
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "nlpub/russe-wsi-kit/initial/data/main/wiki-wiki/train.csv"
)
MODEL_NAME = "intfloat/multilingual-e5-base"

In [4]:
# Оформление таблиц
def show_table(df, formats=None):
    styler = (
        df.style
        .hide(axis="index")
        .set_properties(
            subset=[df.columns[0]],
            **{
                "text-align": "left",
                "font-weight": "bold",
            }
        )
    )

    if len(df.columns) > 1:
        styler = styler.set_properties(
            subset=list(df.columns[1:]),
            **{"text-align": "center"}
        )

    styler = styler.set_table_styles([
        {
            "selector": "th",
            "props": [("text-align", "center")]
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
                ("font-weight", "bold"),
            ]
        },
    ])

    if formats:
        styler = styler.format(formats)

    display(styler)

In [5]:
df = pd.read_csv(DATA_URL, sep="\t")

grouped_df = df.groupby("word")[["word", "context", "gold_sense_id"]]

_data_rows = []
for word, group in grouped_df:
    _data_rows.append({
        "слово": word,
        "контекстов": len(group),
        "значений": group["gold_sense_id"].nunique(),
        "размеры значений": str(dict(group["gold_sense_id"].value_counts().sort_index())),
        "слов в контексте в среднем": round(group["context"].str.split().str.len().mean()),
    })

data_summary = pd.DataFrame(_data_rows)

show_table(
    data_summary,
    {
        "контекстов": "{:.0f}",
        "значений": "{:.0f}",
        "слов в контексте в среднем": "{:.0f}",
    }
)

print(
    f"\nВсего: {len(df)} контекстов, "
    f"{df['word'].nunique()} многозначных слова."
)

слово,контекстов,значений,размеры значений,слов в контексте в среднем
бор,56,2,"{1: np.int64(14), 2: np.int64(42)}",67
замок,138,2,"{1: np.int64(100), 2: np.int64(38)}",73
лук,110,2,"{1: np.int64(65), 2: np.int64(45)}",71
суда,135,2,"{1: np.int64(100), 2: np.int64(35)}",66



Всего: 439 контекстов, 4 многозначных слова.


## 2. Эмбеддинги

Используем intfloat/multilingual-e5-base. Для E5 добавляем префикс query: и сразу нормируем векторы. Считаем эмбеддинги один раз, дальше все алгоритмы работают с одной и той же матрицей — так сравнение будет более качественным.

In [7]:
model = SentenceTransformer(MODEL_NAME)

WORDS = sorted(grouped_df.groups.keys())
GOLD = {
    word: grouped_df.get_group(word)["gold_sense_id"].to_numpy()
    for word in WORDS
}

start = time.perf_counter()

EMB = {}
for word in WORDS:
    contexts = grouped_df.get_group(word)["context"].fillna("").astype(str).tolist()
    texts = ["query: " + text for text in contexts]

    EMB[word] = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

print( f"Размерность: {next(iter(EMB.values())).shape[1]}.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Размерность: 768.


Нужно понять масштаб расстояний. На разных embedding-моделях распределение расстояний может быть совершенно разным, поэтому смотрим квантили косинусных расстояний и дальше часть параметров задаём относительно этих распределений

In [8]:
distance_rows = []

for word in WORDS:
    D = cosine_distances(EMB[word])
    upper = D[np.triu_indices_from(D, k=1)]

    distance_rows.append({
        "слово": word,
        "1%": np.quantile(upper, 0.01),
        "5%": np.quantile(upper, 0.05),
        "10%": np.quantile(upper, 0.10),
        "50%": np.quantile(upper, 0.50),
    })

distance_table = pd.DataFrame(distance_rows)

show_table(
    distance_table,
    {
        "1%": "{:.3f}",
        "5%": "{:.3f}",
        "10%": "{:.3f}",
        "50%": "{:.3f}",
    }
)

слово,1%,5%,10%,50%
бор,0.022,0.106,0.127,0.167
замок,0.075,0.122,0.135,0.167
лук,0.044,0.108,0.120,0.156
суда,0.028,0.108,0.135,0.198


## 3. Функция для экспериментов

Делаем одну общую функцию, которая для каждого слова:
- запускает кластеризацию
- считает ARI
- запоминает число найденных кластеров
- для DBSCAN считает долю объектов, объявленных шумом
- измеряет время

Для случайных алгоритмов можно передать несколько seed, тогда в таблице появится ещё и ARI_std, то есть насколько результат гуляет от запуска к запуску

In [9]:
RESULTS = []

def cos_quantile(X, q):
    D = cosine_distances(X)
    upper = D[np.triu_indices_from(D, k=1)]
    return float(np.quantile(upper, q))

def run_experiment(
    algorithm,
    experiment,
    description,
    factory,
    seeds=(0,),
):
    runs = []
    resolved_values = {}

    for seed in seeds:
        aris = []
        times = []
        clusters = []
        noise = []
        aris_without_noise = []
        converged = []

        for word in WORDS:
            X = EMB[word]
            y_true = GOLD[word]

            clusterer, resolved = factory(X, seed)

            for key, value in resolved.items():
                resolved_values.setdefault(key, []).append(value)

            start = time.perf_counter()

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                labels = clusterer.fit_predict(X)

            times.append(time.perf_counter() - start)
            aris.append(adjusted_rand_score(y_true, labels))

            unique_labels = set(labels.tolist())
            real_clusters = unique_labels - {-1}
            clusters.append(len(real_clusters))

            noise_mask = labels == -1
            noise.append(float(np.mean(noise_mask)))

            keep = ~noise_mask
            if (
                noise_mask.any()
                and keep.sum() > 1
                and len(set(labels[keep].tolist())) > 1
            ):
                aris_without_noise.append(
                    adjusted_rand_score(y_true[keep], labels[keep])
                )

            if isinstance(clusterer, AffinityPropagation):
                converged.append(
                    not any(
                        issubclass(w.category, ConvergenceWarning)
                        for w in caught
                    )
                )

        runs.append({
            "mean_ARI": float(np.mean(aris)),
            "word_ARI": dict(zip(WORDS, aris)),
            "time_sec": float(np.sum(times)),
            "mean_clusters": float(np.mean(clusters)),
            "noise": float(np.mean(noise)),
            "ARI_without_noise": (
                float(np.mean(aris_without_noise))
                if aris_without_noise
                else np.nan
            ),
            "converged": (
                float(np.mean(converged))
                if converged
                else np.nan
            ),
        })

    mean_aris = [run["mean_ARI"] for run in runs]

    row = {
        "algorithm": algorithm,
        "experiment": experiment,
        "params": description,
        "resolved": ", ".join(
            f"{key}≈{np.mean(values):.3g}"
            for key, values in resolved_values.items()
        ),
        "mean_ARI": float(np.mean(mean_aris)),
        "ARI_std": float(np.std(mean_aris)),
        "mean_clusters": float(np.mean([r["mean_clusters"] for r in runs])),
        "noise": float(np.mean([r["noise"] for r in runs])),
        "time_ms": 1000 * float(np.mean([r["time_sec"] for r in runs])),
        "ARI_without_noise": (
            float(np.nanmean([r["ARI_without_noise"] for r in runs]))
            if not np.all(np.isnan([r["ARI_without_noise"] for r in runs]))
            else np.nan
        ),
        "converged": (
            float(np.nanmean([r["converged"] for r in runs]))
            if not np.all(np.isnan([r["converged"] for r in runs]))
            else np.nan
        ),
    }

    for word in WORDS:
        row[f"ARI_{word}"] = float(np.mean([
            r["word_ARI"][word]
            for r in runs
        ]))

    RESULTS.append(row)
    return row


RESULT_FORMATS = {
    "mean_ARI": "{:.3f}",
    "ARI_std": "{:.3f}",
    "mean_clusters": "{:.2f}",
    "noise": "{:.3f}",
    "time_ms": "{:.1f}",
    "ARI_without_noise": "{:.3f}",
    "converged": "{:.2f}",
}

for word in WORDS:
    RESULT_FORMATS[f"ARI_{word}"] = "{:.3f}"

def show_results(algorithm):
    table = pd.DataFrame([
        row for row in RESULTS
        if row["algorithm"] == algorithm
    ])

    columns = [
        "experiment",
        "params",
        "resolved",
        "mean_ARI",
        "ARI_std",
        "mean_clusters",
        "noise",
        "time_ms",
    ]

    if not table["resolved"].astype(bool).any():
        columns.remove("resolved")

    if algorithm == "DBSCAN":
        columns.append("ARI_without_noise")

    if algorithm == "AffinityPropagation":
        columns.append("converged")

    show_table(table[columns], RESULT_FORMATS)

    best = table.loc[table["mean_ARI"].idxmax()]
    print(
        f"\nЛучший результат: {best['experiment']} — "
        f"mean ARI {best['mean_ARI']:.3f}."
    )

Перед основными экспериментами нам полезно убедиться, что ARI ведёт себя ожидаемо. Если раздать объекты по двум кластерам случайно, средний ARI должен быть около нуля:

In [10]:
class RandomLabels:
    def __init__(self, k, seed):
        self.k = k
        self.seed = seed

    def fit_predict(self, X):
        return np.random.default_rng(self.seed).integers(0, self.k, len(X))

baseline = run_experiment(
    "baseline",
    "Random",
    "случайные метки, k=2",
    lambda X, seed: (RandomLabels(2, seed), {}),
    seeds=range(10),
)

baseline_table = pd.DataFrame([baseline])[
    ["experiment", "mean_ARI", "ARI_std"]
]

show_table(baseline_table, RESULT_FORMATS)

experiment,mean_ARI,ARI_std
Random,-0.000,0.005


## 4. KMeans
Заранее задаём k, алгоритм ищет k центров и относит каждый объект к ближайшему центру. Проверяем:
- что происходит, если менять k
- насколько важна инициализация центров
- помогает ли несколько перезапусков n_init

KMeans случайный, поэтому каждую конфигурацию запускаем с 10 разными random_state, так мы видим не один удачный запуск, а средний результат и его разброс

In [11]:
KMEANS_EXPERIMENTS = [
    (
        "KMeans #1",
        "k=2, k-means++, n_init=10",
        lambda X, seed: (
            KMeans(2, init="k-means++", n_init=10, random_state=seed),
            {},
        ),
    ),
    (
        "KMeans #2",
        "k=3",
        lambda X, seed: (KMeans(3, n_init=10, random_state=seed), {}),
    ),
    (
        "KMeans #3",
        "k=4",
        lambda X, seed: (KMeans(4, n_init=10, random_state=seed), {}),
    ),
    (
        "KMeans #4",
        "k=8",
        lambda X, seed: (KMeans(8, n_init=10, random_state=seed), {}),
    ),
    (
        "KMeans #5",
        "k=2, random init, n_init=1",
        lambda X, seed: (
            KMeans(2, init="random", n_init=1, random_state=seed),
            {},
        ),
    ),
    (
        "KMeans #6",
        "k=2, k-means++, n_init=50",
        lambda X, seed: (KMeans(2, n_init=50, random_state=seed), {}),
    ),
]

for experiment in KMEANS_EXPERIMENTS:
    run_experiment(
        "KMeans",
        *experiment,
        seeds=range(10),
    )

show_results("KMeans")

experiment,params,mean_ARI,ARI_std,mean_clusters,noise,time_ms
KMeans #1,"k=2, k-means++, n_init=10",0.641,0.099,2.00,0.000,88.5
KMeans #2,k=3,0.500,0.026,3.00,0.000,110.1
KMeans #3,k=4,0.361,0.037,4.00,0.000,118.5
KMeans #4,k=8,0.202,0.011,8.00,0.000,169.6
KMeans #5,"k=2, random init, n_init=1",0.405,0.174,2.00,0.000,8.1
KMeans #6,"k=2, k-means++, n_init=50",0.669,0.094,2.00,0.000,328.7



Лучший результат: KMeans #6 — mean ARI 0.669.


## 5. DBSCAN
Главные параметры:
- eps — насколько близко точки должны быть друг к другу
- min_samples — сколько соседей нужно, чтобы точка считалась частью плотной области

Eps зависит от масштаба эмбеддингов, поэтому кроме семинарского фиксированного eps=0.1 я хочу попробовать варианты, где он берётся как квантиль реальных попарных расстояний

In [12]:
def dbscan_by_quantile(q, min_samples):
    def factory(X, seed):
        eps = cos_quantile(X, q)
        return (
            DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric="cosine",
            ),
            {"eps": eps},
        )
    return factory


DBSCAN_EXPERIMENTS = [
    (
        "DBSCAN #1",
        "семинар: eps=0.1, min_samples=1, euclidean",
        lambda X, seed: (DBSCAN(eps=0.1, min_samples=1), {}),
    ),
    (
        "DBSCAN #2",
        "cosine, eps=5%-квантиль, min_samples=1",
        dbscan_by_quantile(0.05, 1),
    ),
    (
        "DBSCAN #3",
        "cosine, eps=5%-квантиль, min_samples=5",
        dbscan_by_quantile(0.05, 5),
    ),
    (
        "DBSCAN #4",
        "cosine, eps=10%-квантиль, min_samples=5",
        dbscan_by_quantile(0.10, 5),
    ),
    (
        "DBSCAN #5",
        "cosine, eps=10%-квантиль, min_samples=10",
        dbscan_by_quantile(0.10, 10),
    ),
    (
        "DBSCAN #6",
        "cosine, eps=20%-квантиль, min_samples=10",
        dbscan_by_quantile(0.20, 10),
    ),
]

for experiment in DBSCAN_EXPERIMENTS:
    run_experiment("DBSCAN", *experiment)

show_results("DBSCAN")

experiment,params,resolved,mean_ARI,ARI_std,mean_clusters,noise,time_ms,ARI_without_noise
DBSCAN #1,"семинар: eps=0.1, min_samples=1, euclidean",,0.003,0.000,101.00,0.000,88.1,nan
DBSCAN #2,"cosine, eps=5%-квантиль, min_samples=1",eps≈0.111,0.284,0.000,20.00,0.000,17.4,nan
DBSCAN #3,"cosine, eps=5%-квантиль, min_samples=5",eps≈0.111,0.249,0.000,4.00,0.304,7.5,0.472
DBSCAN #4,"cosine, eps=10%-квантиль, min_samples=5",eps≈0.129,0.055,0.000,1.75,0.133,13.7,0.495
DBSCAN #5,"cosine, eps=10%-квантиль, min_samples=10",eps≈0.129,0.191,0.000,1.75,0.223,10.0,0.797
DBSCAN #6,"cosine, eps=20%-квантиль, min_samples=10",eps≈0.146,-0.030,0.000,1.00,0.029,16.4,nan



Лучший результат: DBSCAN #2 — mean ARI 0.284.


У DBSCAN отдельно показываем ARI_without_noise - это ARI только по тем объектам, которые алгоритм не пометил как шум. Смотреть только на эту колонку нельзя — можно получить красивый результат, просто выбросив большую часть сложных объектов, поэтому рядом всегда остаётся доля шума.

## 6. AffinityPropagation
AffinityPropagation сам выбирает число кластеров и ищет exemplars — реальные объекты, которые становятся представителями групп. Чем выше `preference`, тем охотнее алгоритм создаёт новые кластеры. Абсолютное число опять зависит от масштаба similarities, поэтому я сравниваю не только значение по умолчанию, но и значения относительно реального распределения сходств.

In [13]:
def offdiag(matrix):
    return matrix[np.triu_indices_from(matrix, k=1)]


def ap_factory(damping, preference_fn=None, max_iter=500):
    def factory(X, seed):
        similarity = -euclidean_distances(X, squared=True)

        preference = (
            None
            if preference_fn is None
            else float(preference_fn(similarity))
        )

        info = (
            {}
            if preference is None
            else {"preference": preference}
        )

        return (
            AffinityPropagation(
                damping=damping,
                preference=preference,
                max_iter=max_iter,
                random_state=seed,
            ),
            info,
        )

    return factory

q10_similarity = lambda S: np.quantile(offdiag(S), 0.10)
min_similarity = lambda S: S.min()
min_x2 = lambda S: 2 * S.min()
min_x4 = lambda S: 4 * S.min()
min_x8 = lambda S: 8 * S.min()

AP_EXPERIMENTS = [
    (
        "Affinity #1",
        "damping=0.5, preference по умолчанию",
        ap_factory(0.5),
    ),
    (
        "Affinity #2",
        "damping=0.9, preference по умолчанию",
        ap_factory(0.9),
    ),
    (
        "Affinity #3",
        "damping=0.9, preference=10%-квантиль сходств",
        ap_factory(0.9, q10_similarity),
    ),
    (
        "Affinity #4",
        "damping=0.9, preference=минимум сходств",
        ap_factory(0.9, min_similarity),
    ),
    (
        "Affinity #5",
        "damping=0.9, preference=2 × минимум",
        ap_factory(0.9, min_x2),
    ),
    (
        "Affinity #6",
        "damping=0.9, preference=4 × минимум",
        ap_factory(0.9, min_x4),
    ),
    (
        "Affinity #7",
        "damping=0.9, preference=8 × минимум",
        ap_factory(0.9, min_x8),
    ),
]

for experiment in AP_EXPERIMENTS:
    run_experiment("AffinityPropagation", *experiment)

show_results("AffinityPropagation")

experiment,params,resolved,mean_ARI,ARI_std,mean_clusters,noise,time_ms,converged
Affinity #1,"damping=0.5, preference по умолчанию",,0.058,0.000,25.75,0.000,56.4,1.00
Affinity #2,"damping=0.9, preference по умолчанию",,0.049,0.000,17.50,0.000,43.1,1.00
Affinity #3,"damping=0.9, preference=10%-квантиль сходств",preference≈-0.412,0.070,0.000,23.00,0.000,67.5,1.00
Affinity #4,"damping=0.9, preference=минимум сходств",preference≈-0.497,0.093,0.000,18.75,0.000,52.9,1.00
Affinity #5,"damping=0.9, preference=2 × минимум",preference≈-0.994,0.239,0.000,6.50,0.000,52.7,1.00
Affinity #6,"damping=0.9, preference=4 × минимум",preference≈-1.99,0.495,0.000,3.25,0.000,49.8,1.00
Affinity #7,"damping=0.9, preference=8 × минимум",preference≈-3.98,0.088,0.000,1.25,0.000,40.0,1.00



Лучший результат: Affinity #6 — mean ARI 0.495.


## 7. AgglomerativeClustering
Это иерархическая кластеризация: сначала каждый объект — отдельный кластер, потом ближайшие кластеры постепенно объединяются. Linkage определяет, что именно считать расстоянием между двумя группами, поэтому сначала я сравниваю несколько linkage при одном и том же k=2, а потом отдельно меняю число кластеров. Так проще понять, какой именно параметр влияет на результат, а не менять сразу всё одновременно.

In [14]:
def make_agglomerative(n_clusters, linkage, metric="euclidean"):
    try:
        return AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=linkage,
            metric=metric,
        )
    except TypeError:
        return AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=linkage,
            affinity=metric,
        )

def agglomerative_factory(linkage, k=2, metric="euclidean"):
    return lambda X, seed: (
        make_agglomerative(k, linkage, metric),
        {},
    )

def agglomerative_threshold_factory(fraction):
    def factory(X, seed):
        full_tree = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0,
            linkage="ward",
        ).fit(X)

        threshold = fraction * float(full_tree.distances_.max())

        return (
            AgglomerativeClustering(
                n_clusters=None,
                distance_threshold=threshold,
                linkage="ward",
            ),
            {"threshold": threshold},
        )

    return factory

AGGLOMERATIVE_EXPERIMENTS = [
    (
        "Agglomerative #1",
        "ward, k=2",
        agglomerative_factory("ward", 2),
    ),
    (
        "Agglomerative #2",
        "complete, cosine, k=2",
        agglomerative_factory("complete", 2, "cosine"),
    ),
    (
        "Agglomerative #3",
        "average, cosine, k=2",
        agglomerative_factory("average", 2, "cosine"),
    ),
    (
        "Agglomerative #4",
        "single, cosine, k=2",
        agglomerative_factory("single", 2, "cosine"),
    ),
    (
        "Agglomerative #5",
        "ward, k=3",
        agglomerative_factory("ward", 3),
    ),
    (
        "Agglomerative #6",
        "ward, k=5",
        agglomerative_factory("ward", 5),
    ),
    (
        "Agglomerative #7",
        "ward, threshold=0.7 × высота последнего слияния",
        agglomerative_threshold_factory(0.7),
    ),
]

for experiment in AGGLOMERATIVE_EXPERIMENTS:
    run_experiment("Agglomerative", *experiment)

show_results("Agglomerative")

experiment,params,resolved,mean_ARI,ARI_std,mean_clusters,noise,time_ms
Agglomerative #1,"ward, k=2",,0.558,0.000,2.00,0.000,223.5
Agglomerative #2,"complete, cosine, k=2",,0.356,0.000,2.00,0.000,46.6
Agglomerative #3,"average, cosine, k=2",,0.001,0.000,2.00,0.000,43.2
Agglomerative #4,"single, cosine, k=2",,-0.013,0.000,2.00,0.000,41.9
Agglomerative #5,"ward, k=3",,0.530,0.000,3.00,0.000,10.8
Agglomerative #6,"ward, k=5",,0.332,0.000,5.00,0.000,10.9
Agglomerative #7,"ward, threshold=0.7 × высота последнего слияния",threshold≈1.31,0.444,0.000,3.25,0.000,10.6



Лучший результат: Agglomerative #1 — mean ARI 0.558.


## 8. Дополнительный алгоритм: SpectralClustering
Он смотрит не столько на центры или плотность, сколько на граф связей между объектами: похожие контексты соединяются друг с другом, а потом алгоритм пытается разрезать этот граф там, где связей мало. Поэтому он может находить группы более сложной формы, чем KMeans. Проверяем:
- сколько соседей использовать в графе
- nearest_neighbors против RBF-ядра
- насколько чувствителен RBF к gamma
- что будет, если изменить k

SpectralClustering может зависеть от случайной инициализации, поэтому используем несколько seed

In [20]:
def spectral_factory(
    k=2,
    affinity="nearest_neighbors",
    n_neighbors=10,
    gamma_mode=None,
):
    def factory(X, seed):
        kwargs = {
            "n_clusters": k,
            "affinity": affinity,
            "random_state": seed,
            "assign_labels": "kmeans",
        }
        info = {}

        if affinity == "nearest_neighbors":
            kwargs["n_neighbors"] = n_neighbors

        if affinity == "rbf":
            if gamma_mode is None:
                gamma = 1.0
            else:
                d2 = euclidean_distances(X, squared=True)
                upper = d2[np.triu_indices_from(d2, k=1)]
                median_d2 = float(np.median(upper))
                gamma = gamma_mode / median_d2

            kwargs["gamma"] = gamma
            info["gamma"] = gamma

        return SpectralClustering(**kwargs), info

    return factory


SPECTRAL_EXPERIMENTS = [
    (
        "Spectral #1",
        "k=2, nearest_neighbors=10",
        spectral_factory(2, "nearest_neighbors", 10),
    ),
    (
        "Spectral #2",
        "k=2, nearest_neighbors=5",
        spectral_factory(2, "nearest_neighbors", 5),
    ),
    (
        "Spectral #3",
        "k=2, nearest_neighbors=30",
        spectral_factory(2, "nearest_neighbors", 30),
    ),
    (
        "Spectral #4",
        "k=2, rbf, gamma=1",
        spectral_factory(2, "rbf", gamma_mode=None),
    ),
    (
        "Spectral #5",
        "k=2, rbf, gamma=1/median(d²)",
        spectral_factory(2, "rbf", gamma_mode=1.0),
    ),
    (
        "Spectral #6",
        "k=2, rbf, gamma=10/median(d²)",
        spectral_factory(2, "rbf", gamma_mode=10.0),
    ),
    (
        "Spectral #7",
        "k=3, nearest_neighbors=10",
        spectral_factory(3, "nearest_neighbors", 10),
    ),
]

for experiment in SPECTRAL_EXPERIMENTS:
    run_experiment(
        "Spectral",
        *experiment,
        seeds=range(5),
    )

show_results("Spectral")

experiment,params,resolved,mean_ARI,ARI_std,mean_clusters,noise,time_ms
Spectral #1,"k=2, nearest_neighbors=10",,0.591,0.000,2.00,0.000,114.0
Spectral #2,"k=2, nearest_neighbors=5",,0.678,0.000,2.00,0.000,54.3
Spectral #3,"k=2, nearest_neighbors=30",,0.518,0.003,2.00,0.000,61.0
Spectral #4,"k=2, rbf, gamma=1",gamma≈1,0.558,0.000,2.00,0.000,49.8
Spectral #5,"k=2, rbf, gamma=1/median(d²)",gamma≈2.93,0.555,0.000,2.00,0.000,45.9
Spectral #6,"k=2, rbf, gamma=10/median(d²)",gamma≈29.3,-0.029,0.000,2.00,0.000,32.5
Spectral #7,"k=3, nearest_neighbors=10",,0.503,0.000,3.00,0.000,41.1



Лучший результат: Spectral #2 — mean ARI 0.678.


## 9. Сравниваю лучшие результаты
Теперь беру лучший ARI внутри каждого алгоритма и лучший вариант выбираю по gold-разметке. Таблица показывает, что алгоритм может дать при удачной настройке, а не качество полностью автоматического метода без разметки

In [22]:
results_df = pd.DataFrame(RESULTS)
results_df = results_df[
    results_df["algorithm"] != "baseline"
].copy()

ALGORITHM_ORDER = [
    "KMeans",
    "DBSCAN",
    "AffinityPropagation",
    "Agglomerative",
    "Spectral",
]

parameter_info = {
    "KMeans": {
        "понятность параметров": "высокая",
        "комментарий": (
            "параметры простые, но число кластеров "
            "нужно знать или подбирать"
        ),
    },
    "DBSCAN": {
        "понятность параметров": "средняя",
        "комментарий": (
            "eps понятен по смыслу, но сильно зависит "
            "от масштаба embeddings"
        ),
    },
    "AffinityPropagation": {
        "понятность параметров": "низкая",
        "комментарий": (
            "preference сильно влияет на число кластеров "
            "и его трудно выбрать заранее"
        ),
    },
    "Agglomerative": {
        "понятность параметров": "средняя–высокая",
        "комментарий": (
            "linkage понятен по смыслу, но его выбор "
            "сильно влияет на результат"
        ),
    },
    "Spectral": {
        "понятность параметров": "средняя",
        "комментарий": (
            "идея графа понятна, но affinity и параметры "
            "графа требуют дополнительной настройки"
        ),
    },
}

available_algorithms = set(
    results_df["algorithm"]
)

missing_algorithms = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm not in available_algorithms
]

best_rows = []

for algorithm in ALGORITHM_ORDER:
    part = results_df[
        results_df["algorithm"] == algorithm
    ]

    if part.empty:
        continue

    best = part.loc[
        part["mean_ARI"].idxmax()
    ]

    best_rows.append({
        "алгоритм": algorithm,
        "лучший эксперимент": best["experiment"],
        "mean ARI": best["mean_ARI"],
        "ARI std": best["ARI_std"],
        "кластеров": best["mean_clusters"],
        "время, мс": best["time_ms"],
        "минимальный ARI": part["mean_ARI"].min(),
        "медианный ARI": part["mean_ARI"].median(),
        "максимальный ARI": part["mean_ARI"].max(),
        "понятность параметров":
            parameter_info[algorithm]["понятность параметров"],
        "комментарий":
            parameter_info[algorithm]["комментарий"],
    })

best_summary = pd.DataFrame(best_rows)

show_table(
    best_summary,
    {
        "mean ARI": "{:.3f}",
        "ARI std": "{:.3f}",
        "кластеров": "{:.2f}",
        "время, мс": "{:.1f}",
        "минимальный ARI": "{:.3f}",
        "медианный ARI": "{:.3f}",
        "максимальный ARI": "{:.3f}",
    }
)

алгоритм,лучший эксперимент,mean ARI,ARI std,кластеров,"время, мс",минимальный ARI,медианный ARI,максимальный ARI,понятность параметров,комментарий
KMeans,KMeans #6,0.669,0.094,2.00,328.7,0.202,0.453,0.669,высокая,"параметры простые, но число кластеров нужно знать или подбирать"
DBSCAN,DBSCAN #2,0.284,0.000,20.00,17.4,-0.030,0.123,0.284,средняя,"eps понятен по смыслу, но сильно зависит от масштаба embeddings"
AffinityPropagation,Affinity #6,0.495,0.000,3.25,49.8,0.049,0.088,0.495,низкая,preference сильно влияет на число кластеров и его трудно выбрать заранее
Agglomerative,Agglomerative #1,0.558,0.000,2.00,223.5,-0.013,0.356,0.558,средняя–высокая,"linkage понятен по смыслу, но его выбор сильно влияет на результат"
Spectral,Spectral #2,0.678,0.000,2.00,54.3,-0.029,0.555,0.678,средняя,"идея графа понятна, но affinity и параметры графа требуют дополнительной настройки"


Если максимум высокий, а остальные настройки проваливаются, можно сделать вывод, что алгоритм очень чувствителен к гиперпараметрам и без разметки подобрать их будет сложно.

## 10. Насколько понятны гиперпараметры?
Это практическая оценка: нужно зафиксировать, что делать, когда gold-разметки нет:

In [24]:
parameter_table = pd.DataFrame([
    {
        "алгоритм": "KMeans",
        "главные параметры": "n_clusters, init, n_init",
        "понятность": "высокая",
        "комментарий": "параметры простые, но число кластеров нужно знать или подбирать",
    },
    {
        "алгоритм": "DBSCAN",
        "главные параметры": "eps, min_samples",
        "понятность": "средняя",
        "комментарий": "смысл понятен, но eps сильно зависит от масштаба эмбеддингов",
    },
    {
        "алгоритм": "AffinityPropagation",
        "главные параметры": "preference, damping",
        "понятность": "низкая",
        "комментарий": "preference сильно влияет на число кластеров и его трудно выбрать заранее",
    },
    {
        "алгоритм": "Agglomerative",
        "главные параметры": "linkage, n_clusters / threshold",
        "понятность": "средняя–высокая",
        "комментарий": "linkage интерпретируем, но результат от его выбора меняется сильно",
    },
    {
        "алгоритм": "Spectral",
        "главные параметры": "n_clusters, affinity, n_neighbors / gamma",
        "понятность": "средняя",
        "комментарий": "идея графа понятна, но параметры графа требуют настройки",
    },
])

show_table(parameter_table)

алгоритм,главные параметры,понятность,комментарий
KMeans,"n_clusters, init, n_init",высокая,"параметры простые, но число кластеров нужно знать или подбирать"
DBSCAN,"eps, min_samples",средняя,"смысл понятен, но eps сильно зависит от масштаба эмбеддингов"
AffinityPropagation,"preference, damping",низкая,preference сильно влияет на число кластеров и его трудно выбрать заранее
Agglomerative,"linkage, n_clusters / threshold",средняя–высокая,"linkage интерпретируем, но результат от его выбора меняется сильно"
Spectral,"n_clusters, affinity, n_neighbors / gamma",средняя,"идея графа понятна, но параметры графа требуют настройки"


## 11. Выводы

In [26]:
best_overall = best_summary.loc[best_summary["mean ARI"].idxmax()]

largest_n = scaling_df["N"].max()
fastest_large = (
    scaling_df[scaling_df["N"] == largest_n]
    .sort_values("время, сек")
    .iloc[0]
)

print(
    f"Лучший ARI: {best_overall['алгоритм']} — "
    f"{best_overall['mean ARI']:.3f} "
    f"({best_overall['лучший эксперимент']})."
)

print(
    f"Самый быстрый на N={largest_n}: "
    f"{fastest_large['алгоритм']} — "
    f"{fastest_large['время, сек']:.3f} сек."
)

Лучший ARI: Spectral — 0.678 (Spectral #2).
Самый быстрый на N=3200: DBSCAN — 0.193 сек.
